# StudyF 原始 EEG 检查：一个被试、一个 session

本 notebook 只做 raw data inspection，不进行滤波、ICA、伪迹修复、epoch 切分或 target/non-target 解码。

当前范围：

- 被试：`F_03`
- Session：`SE001`
- 数据：`Train/CBCol` 下的 EDF 文件

bigP3BCI 使用 EDF 格式，因此使用 `mne.io.read_raw_edf()`，而不是教程示例中的 `read_raw_fif()`。


In [2]:
%matplotlib qt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

PROJECT_ROOT=Path.cwd()
while PROJECT_ROOT!=PROJECT_ROOT.parent and not ((PROJECT_ROOT/'README.md').exists() or (PROJECT_ROOT/'.git').exists()):
    PROJECT_ROOT=PROJECT_ROOT.parent
if not ((PROJECT_ROOT/'README.md').exists() or (PROJECT_ROOT/'.git').exists()):
    raise FileNotFoundError('请在项目目录或子目录启动 Jupyter。')

SUBJECT_ID='F_03'
SESSION_ID='SE001'
TRAIN_ROOT=PROJECT_ROOT/'data'/'raw'/'bigP3BCI-data'/'StudyF'/SUBJECT_ID/SESSION_ID/'Train'/'CBCol'
if not TRAIN_ROOT.is_dir():
    raise FileNotFoundError(f'找不到：{TRAIN_ROOT}')

train_files=sorted(TRAIN_ROOT.glob('*.edf'))
if not train_files:
    raise FileNotFoundError(f'目录中没有 EDF 文件：{TRAIN_ROOT}')

raw_train={path.name:mne.io.read_raw_edf(path,preload=False,infer_types=True,verbose=False) for path in train_files}

def assign_bigp3bci_channel_types(raw):
    import re
    event_names={'StimulusBegin','StimulusType','StimulusCode','CurrentTarget','FakeFeedback','DisplayResults','DisplayResult','SelectedTarget','SelectedRow','SelectedColumn','PhaseInSequence'}
    channel_types={}
    for name in raw.ch_names:
        if name in event_names:
            channel_types[name]='stim'
        elif re.match(r'^.+_\d+_\d+$',name):
            channel_types[name]='misc'
        elif name.upper().startswith('EOG'):
            channel_types[name]='eog'
    if channel_types:
        raw.set_channel_types(channel_types,on_unit_change='ignore',verbose=False)
    return raw

raw_train={name:assign_bigp3bci_channel_types(raw) for name,raw in raw_train.items()}
print('MNE:',mne.__version__)
print('Subject:',SUBJECT_ID)
print('Session:',SESSION_ID)
print('EDF files:',len(raw_train))
print([path.name for path in train_files])
print('Corrected channel types:', dict(zip(next(iter(raw_train.values())).ch_names, next(iter(raw_train.values())).get_channel_types())))


MNE: 1.12.1
Subject: F_03
Session: SE001
EDF files: 3
['F_03_SE001_CBCol_Train01.edf', 'F_03_SE001_CBCol_Train02.edf', 'F_03_SE001_CBCol_Train03.edf']
Corrected channel types: {'EEG_F3': 'eeg', 'EEG_Fz': 'eeg', 'EEG_F4': 'eeg', 'EEG_T7': 'eeg', 'EEG_C3': 'eeg', 'EEG_Cz': 'eeg', 'EEG_C4': 'eeg', 'EEG_T8': 'eeg', 'EEG_CP3': 'eeg', 'EEG_CP4': 'eeg', 'EEG_P3': 'eeg', 'EEG_Pz': 'eeg', 'EEG_P4': 'eeg', 'EEG_PO7': 'eeg', 'EEG_PO8': 'eeg', 'EEG_Oz': 'eeg', 'A_1_1': 'misc', 'B_1_2': 'misc', 'C_1_3': 'misc', 'D_1_4': 'misc', 'E_1_5': 'misc', 'F_1_6': 'misc', 'G_2_1': 'misc', 'H_2_2': 'misc', 'I_2_3': 'misc', 'J_2_4': 'misc', 'K_2_5': 'misc', 'L_2_6': 'misc', 'M_3_1': 'misc', 'N_3_2': 'misc', 'O_3_3': 'misc', 'P_3_4': 'misc', 'Q_3_5': 'misc', 'R_3_6': 'misc', 'S_4_1': 'misc', 'T_4_2': 'misc', 'U_4_3': 'misc', 'V_4_4': 'misc', 'W_4_5': 'misc', 'X_4_6': 'misc', 'Y_5_1': 'misc', 'Z_5_2': 'misc', 'Sp_5_3': 'misc', '1_5_4': 'misc', '2_5_5': 'misc', '3_5_6': 'misc', '4_6_1': 'misc', '5_6_2': 'misc', '6

C:\Users\78431\AppData\Local\Temp\ipykernel_34568\1464875254.py:24: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw_train={path.name:mne.io.read_raw_edf(path,preload=False,infer_types=True,verbose=False) for path in train_files}
C:\Users\78431\AppData\Local\Temp\ipykernel_34568\1464875254.py:24: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw_train={path.name:mne.io.read_raw_edf(path,preload=False,infer_types=True,verbose=False) for path in train_files}
C:\Users\78431\AppData\Local\Temp\ipykernel_34568\1464875254.py:24: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw_train={path.name:mne.io.read_raw_edf(path,preload=False,infer_types=True,verbose=False) for path in train_files}
C:\Users\78431\AppData\Local\Temp\ipykernel_34568\1464875254.py:24: RuntimeWarning: Channels contain different lowpass filters. Lowest filter

Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none


In [3]:
# 明确修正 bigP3BCI EDF 的通道类型
import re

EVENT_CHANNELS = {
    'StimulusBegin', 'StimulusType', 'StimulusCode', 'CurrentTarget',
    'FakeFeedback', 'DisplayResults', 'DisplayResult', 'SelectedTarget',
    'SelectedRow', 'SelectedColumn', 'PhaseInSequence'
}

for filename, raw in raw_train.items():
    type_map = {}
    for name in raw.ch_names:
        if name in EVENT_CHANNELS:
            type_map[name] = 'stim'
        elif re.match(r'^.+_\d+_\d+$', name):
            type_map[name] = 'misc'
        elif name.upper().startswith('EOG'):
            type_map[name] = 'eog'
    raw.set_channel_types(type_map, on_unit_change='ignore', verbose=False)

channel_type_check = pd.concat([
    pd.DataFrame({
        'file': filename,
        'channel': raw.ch_names,
        'type': raw.get_channel_types(),
    })
    for filename, raw in raw_train.items()
], ignore_index=True)

display(channel_type_check)
print('Stim/event types:', channel_type_check[channel_type_check['channel'].isin(EVENT_CHANNELS)]['type'].value_counts().to_dict())
print('EEG count by file:')
display(channel_type_check.groupby(['file', 'type']).size().unstack(fill_value=0))


,file,channel,type
0,F_03_SE001_CBCol_Train01.edf,EEG_F3,eeg
1,F_03_SE001_CBCol_Train01.edf,EEG_Fz,eeg
2,F_03_SE001_CBCol_Train01.edf,EEG_F4,eeg
3,F_03_SE001_CBCol_Train01.edf,EEG_T7,eeg
4,F_03_SE001_CBCol_Train01.edf,EEG_C3,eeg
...,...,...,...
181,F_03_SE001_CBCol_Train03.edf,StimulusBegin,stim
182,F_03_SE001_CBCol_Train03.edf,StimulusCode,stim
183,F_03_SE001_CBCol_Train03.edf,CurrentTarget,stim
184,F_03_SE001_CBCol_Train03.edf,FakeFeedback,stim


Stim/event types: {'stim': 30}
EEG count by file:


type,eeg,misc,stim
file,,,
F_03_SE001_CBCol_Train01.edf,16,36,10
F_03_SE001_CBCol_Train02.edf,16,36,10
F_03_SE001_CBCol_Train03.edf,16,36,10


## 1. 查看数据概况

先确认每个 EDF 都能正常读取，并记录：

- MNE 的 `Raw` 摘要；
- `raw.info`；
- 通道名称和数量；
- 采样率；
- 记录时长；
- 已标记的 bad channels；
- 通道类型。


In [4]:
for filename, raw in raw_train.items():
    print('\n' + '=' * 90)
    print(filename)
    print(raw)
    print('\nraw.info:')
    print(raw.info)
    print('\nNumber of channels:', len(raw.ch_names))
    print('Sampling frequency:', raw.info['sfreq'], 'Hz')
    print('Duration:', raw.times[-1], 'seconds')
    print('Bad channels:', raw.info['bads'])
    print('Channel types:')
    display(pd.DataFrame({'channel':raw.ch_names,'type':raw.get_channel_types()}))



F_03_SE001_CBCol_Train01.edf
<RawEDF | F_03_SE001_CBCol_Train01.edf, 62 x 55808 (218.0 s), ~51 KiB, data not loaded>

raw.info:
<Info | 8 non-empty values
 bads: []
 ch_names: EEG_F3, EEG_Fz, EEG_F4, EEG_T7, EEG_C3, EEG_Cz, EEG_C4, EEG_T8, ...
 chs: 16 EEG, 36 misc, 10 Stimulus
 custom_ref_applied: False
 highpass: 58.0 Hz
 lowpass: 62.0 Hz
 meas_date: 2020-01-01 00:00:00 UTC
 nchan: 62
 projs: []
 sfreq: 256.0 Hz
 subject_info: <subject_info | his_id: F_03, sex: 2, first_name: X, last_name: X, birthday: 1976-01-01>
>

Number of channels: 62
Sampling frequency: 256.0 Hz
Duration: 217.99609375 seconds
Bad channels: []
Channel types:


,channel,type
0,EEG_F3,eeg
1,EEG_Fz,eeg
2,EEG_F4,eeg
3,EEG_T7,eeg
4,EEG_C3,eeg
...,...,...
57,StimulusBegin,stim
58,StimulusCode,stim
59,CurrentTarget,stim
60,FakeFeedback,stim



F_03_SE001_CBCol_Train02.edf
<RawEDF | F_03_SE001_CBCol_Train02.edf, 62 x 55808 (218.0 s), ~51 KiB, data not loaded>

raw.info:
<Info | 8 non-empty values
 bads: []
 ch_names: EEG_F3, EEG_Fz, EEG_F4, EEG_T7, EEG_C3, EEG_Cz, EEG_C4, EEG_T8, ...
 chs: 16 EEG, 36 misc, 10 Stimulus
 custom_ref_applied: False
 highpass: 58.0 Hz
 lowpass: 62.0 Hz
 meas_date: 2020-01-01 00:00:00 UTC
 nchan: 62
 projs: []
 sfreq: 256.0 Hz
 subject_info: <subject_info | his_id: F_03, sex: 2, first_name: X, last_name: X, birthday: 1976-01-01>
>

Number of channels: 62
Sampling frequency: 256.0 Hz
Duration: 217.99609375 seconds
Bad channels: []
Channel types:


,channel,type
0,EEG_F3,eeg
1,EEG_Fz,eeg
2,EEG_F4,eeg
3,EEG_T7,eeg
4,EEG_C3,eeg
...,...,...
57,StimulusBegin,stim
58,StimulusCode,stim
59,CurrentTarget,stim
60,FakeFeedback,stim



F_03_SE001_CBCol_Train03.edf
<RawEDF | F_03_SE001_CBCol_Train03.edf, 62 x 55808 (218.0 s), ~51 KiB, data not loaded>

raw.info:
<Info | 8 non-empty values
 bads: []
 ch_names: EEG_F3, EEG_Fz, EEG_F4, EEG_T7, EEG_C3, EEG_Cz, EEG_C4, EEG_T8, ...
 chs: 16 EEG, 36 misc, 10 Stimulus
 custom_ref_applied: False
 highpass: 58.0 Hz
 lowpass: 62.0 Hz
 meas_date: 2020-01-01 00:00:00 UTC
 nchan: 62
 projs: []
 sfreq: 256.0 Hz
 subject_info: <subject_info | his_id: F_03, sex: 2, first_name: X, last_name: X, birthday: 1976-01-01>
>

Number of channels: 62
Sampling frequency: 256.0 Hz
Duration: 217.99609375 seconds
Bad channels: []
Channel types:


,channel,type
0,EEG_F3,eeg
1,EEG_Fz,eeg
2,EEG_F4,eeg
3,EEG_T7,eeg
4,EEG_C3,eeg
...,...,...
57,StimulusBegin,stim
58,StimulusCode,stim
59,CurrentTarget,stim
60,FakeFeedback,stim


## 2. 画一小段连续 EEG 信号

按照原始数据检查的要求，这里只画约 20 秒的 EEG 通道。暂时不滤波、不重参考，也不做 DC correction；先观察是否存在：

- 平直通道；
- 突然的大幅跳变；
- 明显高频噪声；
- 某个通道长期远高于其他通道；
- 明显低频漂移。

字符呈现通道、Stimulus/event 通道和反馈通道不参与 EEG 波形检查。


In [5]:
def eeg_channels_only(raw):
    return [name for name, kind in zip(raw.ch_names, raw.get_channel_types()) if kind == 'eeg' and name not in raw.info['bads']]

for filename, raw in raw_train.items():
    eeg=eeg_channels_only(raw)
    print(filename, 'EEG channels:', len(eeg))
    if eeg:
        raw.plot(duration=min(20,raw.times[-1]),start=0,picks=eeg,n_channels=len(eeg),remove_dc=False,scalings='auto',title=f'{filename}: raw EEG, first 20 seconds',show=True)


F_03_SE001_CBCol_Train01.edf EEG channels: 16
Using matplotlib as 2D backend.
F_03_SE001_CBCol_Train02.edf EEG channels: 16
F_03_SE001_CBCol_Train03.edf EEG channels: 16


## 3. 查看未滤波频谱

这里直接对原始 EEG 计算 PSD，不先滤波。重点观察：

- 能量主要分布在哪些频段；
- 低频能量是否异常突出；
- 50/60 Hz 附近是否有明显电源线噪声峰；
- 不同 EEG 通道的频谱是否明显异常。

此步骤只用于检查，不在这里应用 notch 或 band-pass filter。


In [6]:
for filename, raw in raw_train.items():
    eeg=eeg_channels_only(raw)
    if not eeg:
        continue
    spectrum=raw.compute_psd(fmin=0.1,fmax=min(250,raw.info['sfreq']/2-1),picks=eeg,reject_by_annotation=False,exclude='bads',verbose=False)
    spectrum.plot(average=True,amplitude=False,show=True)
    print(filename, ': unfiltered PSD displayed')


Plotting power spectral density (dB=True).
F_03_SE001_CBCol_Train01.edf : unfiltered PSD displayed
Plotting power spectral density (dB=True).
F_03_SE001_CBCol_Train02.edf : unfiltered PSD displayed
Plotting power spectral density (dB=True).
F_03_SE001_CBCol_Train03.edf : unfiltered PSD displayed


## 4. 查看 annotations 和原始事件信息

先查看 `raw.annotations` 的数量、时间和 description。这里不把 annotation 自动转换成事件，也不解码 `StimulusBegin`、`StimulusType` 或 `StimulusCode`。

同时列出可能的 trigger/event/status/target 通道名称，后续再单独核对它们的编码定义。


In [7]:
EVENT_WORDS=('stimulus','event','trigger','status','marker','target','feedback','display','phase','selected')

for filename, raw in raw_train.items():
    print('\n' + '=' * 90)
    print(filename)
    print('Number of annotations:', len(raw.annotations))

    if len(raw.annotations):
        annotation_table=pd.DataFrame({'onset':raw.annotations.onset,'duration':raw.annotations.duration,'description':raw.annotations.description})
        display(annotation_table)
    else:
        print('No annotations found.')

    possible_channels=[name for name in raw.ch_names if any(word in name.lower() for word in EVENT_WORDS)]
    print('Possible trigger/event channels:', possible_channels or 'none found')
    print('All non-EEG channels:')
    display(pd.DataFrame({'channel':raw.ch_names,'type':raw.get_channel_types()}).query("type != 'eeg'"))



F_03_SE001_CBCol_Train01.edf
Number of annotations: 1


,onset,duration,description
0,0.0,0.0,Begin recording


Possible trigger/event channels: ['StimulusType', 'SelectedTarget', 'SelectedRow', 'SelectedColumn', 'PhaseInSequence', 'StimulusBegin', 'StimulusCode', 'CurrentTarget', 'FakeFeedback', 'DisplayResults']
All non-EEG channels:


,channel,type
16,A_1_1,misc
17,B_1_2,misc
18,C_1_3,misc
19,D_1_4,misc
20,E_1_5,misc
21,F_1_6,misc
22,G_2_1,misc
23,H_2_2,misc
24,I_2_3,misc
25,J_2_4,misc



F_03_SE001_CBCol_Train02.edf
Number of annotations: 1


,onset,duration,description
0,0.0,0.0,Begin recording


... (2 outputs left out)


F_03_SE001_CBCol_Train03.edf
Number of annotations: 1


,onset,duration,description
0,0.0,0.0,Begin recording


Possible trigger/event channels: ['StimulusType', 'SelectedTarget', 'SelectedRow', 'SelectedColumn', 'PhaseInSequence', 'StimulusBegin', 'StimulusCode', 'CurrentTarget', 'FakeFeedback', 'DisplayResults']
All non-EEG channels:


,channel,type
16,A_1_1,misc
17,B_1_2,misc
18,C_1_3,misc
19,D_1_4,misc
20,E_1_5,misc
21,F_1_6,misc
22,G_2_1,misc
23,H_2_2,misc
24,I_2_3,misc
25,J_2_4,misc


## 5. 保留原始数据不变

当前 notebook 中的 `raw_train` 使用 `preload=False` 读取，且没有对原始对象执行滤波、裁剪、重参考、坏道插值或 ICA。

如果后续需要尝试裁剪、选通道或滤波，必须先复制：


In [8]:
# 示例：只对副本进行探索，不改变 raw_train 中的原始 Raw
example_filename=next(iter(raw_train))
raw_original=raw_train[example_filename]
raw_copy=raw_original.copy()

# 后续探索应在 raw_copy 上进行，例如：
# raw_copy.crop(tmax=60)
# raw_copy.pick(eeg_channels_only(raw_copy))
# raw_copy.filter(l_freq=0.1, h_freq=40.0)

print('Original duration:', raw_original.times[-1])
print('Copy duration:', raw_copy.times[-1])
print('Original object remains unchanged.')


Original duration: 217.99609375
Copy duration: 217.99609375
Original object remains unchanged.


## 当前阶段结论记录

完成上述图形和表格检查后，再记录：

- 文件是否均能读取；
- 通道数量和采样率是否一致；
- 是否存在已有 bad channels；
- 是否存在明显低频漂移；
- 是否存在 50/60 Hz 峰；
- annotations 是否为空；
- 哪些通道可能承担 trigger/event/status 作用。

本 notebook 到这里结束，不在本阶段解释事件编码或生成 P300 标签。


## 6. 导出完整通道表

下面不截断显示所有通道，并将通道清单导出为 CSV。表中保留文件名、通道序号、通道名称、MNE 识别类型、采样率、记录时长以及 raw.info 中的原始采集滤波元数据。


In [9]:
channel_rows=[]
for filename, raw in raw_train.items():
    for index, (name, kind) in enumerate(zip(raw.ch_names, raw.get_channel_types())):
        channel_rows.append({
            'file': filename,
            'channel_index': index,
            'channel_name': name,
            'mne_type': kind,
            'sfreq_hz': raw.info['sfreq'],
            'duration_sec': raw.times[-1],
            'raw_highpass_hz': raw.info['highpass'],
            'raw_lowpass_hz': raw.info['lowpass'],
            'is_bad': name in raw.info['bads'],
        })

channel_inventory = pd.DataFrame(channel_rows)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

display(channel_inventory)

export_dir = PROJECT_ROOT / 'outputs' / 'raw_inspection'
export_dir.mkdir(parents=True, exist_ok=True)
export_path = export_dir / f'{SUBJECT_ID}_{SESSION_ID}_channel_inventory.csv'
channel_inventory.to_csv(export_path, index=False, encoding='utf-8-sig')
print(f'完整通道表已导出：{export_path}')


,file,channel_index,channel_name,mne_type,sfreq_hz,duration_sec,raw_highpass_hz,raw_lowpass_hz,is_bad
0,F_03_SE001_CBCol_Train01.edf,0,EEG_F3,eeg,256.0,217.996094,58.0,62.0,False
1,F_03_SE001_CBCol_Train01.edf,1,EEG_Fz,eeg,256.0,217.996094,58.0,62.0,False
2,F_03_SE001_CBCol_Train01.edf,2,EEG_F4,eeg,256.0,217.996094,58.0,62.0,False
3,F_03_SE001_CBCol_Train01.edf,3,EEG_T7,eeg,256.0,217.996094,58.0,62.0,False
4,F_03_SE001_CBCol_Train01.edf,4,EEG_C3,eeg,256.0,217.996094,58.0,62.0,False
5,F_03_SE001_CBCol_Train01.edf,5,EEG_Cz,eeg,256.0,217.996094,58.0,62.0,False
6,F_03_SE001_CBCol_Train01.edf,6,EEG_C4,eeg,256.0,217.996094,58.0,62.0,False
7,F_03_SE001_CBCol_Train01.edf,7,EEG_T8,eeg,256.0,217.996094,58.0,62.0,False
8,F_03_SE001_CBCol_Train01.edf,8,EEG_CP3,eeg,256.0,217.996094,58.0,62.0,False
9,F_03_SE001_CBCol_Train01.edf,9,EEG_CP4,eeg,256.0,217.996094,58.0,62.0,False


完整通道表已导出：d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection\F_03_SE001_channel_inventory.csv


## 7. 通道类型与用途简表

下面按通道名称去重，只保留一行，列出 MNE 类型和在 bigP3BCI 中的简易用途说明。


## 8. Stimulus/event 通道审计

本节不预先假设所有通道都有效，而是逐个检查它们的取值、非零比例、唯一值和 stimulus onset 对齐值。

保留策略：

- 核心标签：`StimulusBegin`、`StimulusType`、`CurrentTarget`
- 刺激顺序和阶段：`StimulusCode`、`PhaseInSequence`
- Test 反馈结果：`SelectedTarget`、`SelectedRow`、`SelectedColumn`、`DisplayResults`、`FakeFeedback`
- 全零通道不用于建模，但保留在审计表中记录。


In [10]:
STIMULUS_CHANNELS = [
    'StimulusBegin', 'StimulusType', 'StimulusCode', 'CurrentTarget',
    'PhaseInSequence', 'SelectedTarget', 'SelectedRow', 'SelectedColumn',
    'DisplayResults', 'DisplayResult', 'FakeFeedback',
]

CORE_LABEL_CHANNELS = ['StimulusBegin', 'StimulusType', 'CurrentTarget']
SEQUENCE_CHANNELS = ['StimulusCode', 'PhaseInSequence']
FEEDBACK_CHANNELS = ['SelectedTarget', 'SelectedRow', 'SelectedColumn', 'DisplayResults', 'DisplayResult', 'FakeFeedback']

channel_audit_rows = []
onset_event_tables = []

for filename, raw in raw_train.items():
    present = [name for name in STIMULUS_CHANNELS if name in raw.ch_names]

    onset_samples = np.array([], dtype=int)
    if 'StimulusBegin' in raw.ch_names:
        begin = raw.get_data(picks=['StimulusBegin'], verbose=False)[0]
        onset_samples = np.flatnonzero(
            np.diff((begin > 0.5).astype(int), prepend=0) == 1
        )

    for name in present:
        values = raw.get_data(picks=[name], verbose=False)[0]
        unique_values, counts = np.unique(values, return_counts=True)
        nonzero = values != 0
        transitions = np.count_nonzero(np.diff(values) != 0)

        channel_audit_rows.append({
            'file': filename,
            'channel': name,
            'role': (
                'core label' if name in CORE_LABEL_CHANNELS
                else 'sequence/phase' if name in SEQUENCE_CHANNELS
                else 'feedback/result' if name in FEEDBACK_CHANNELS
                else 'other'
            ),
            'min': values.min(),
            'max': values.max(),
            'std': values.std(),
            'nonzero_count': int(nonzero.sum()),
            'nonzero_fraction': float(nonzero.mean()),
            'unique_count': len(unique_values),
            'unique_values_first': unique_values[:20].tolist(),
            'transition_count': int(transitions),
            'all_zero': bool(np.all(values == 0)),
            'values_at_onset_first': values[onset_samples[:10]].tolist() if len(onset_samples) else [],
        })

    if len(onset_samples):
        onset_table = pd.DataFrame({
            'file': filename,
            'sample': onset_samples,
            'time_sec': onset_samples / raw.info['sfreq'],
        })
        for name in present:
            values = raw.get_data(picks=[name], verbose=False)[0]
            onset_table[name] = values[onset_samples]
        onset_event_tables.append(onset_table)

channel_audit = pd.DataFrame(channel_audit_rows)
onset_events = pd.concat(onset_event_tables, ignore_index=True) if onset_event_tables else pd.DataFrame()

print('Channel audit:')
display(channel_audit)

print('All-zero channels by file:')
display(channel_audit[channel_audit['all_zero']][['file', 'channel', 'role']])

print('Values at first stimulus onsets:')
display(onset_events.head(20))

# Export both tables
export_dir = PROJECT_ROOT / 'outputs' / 'raw_inspection'
export_dir.mkdir(parents=True, exist_ok=True)
channel_audit.to_csv(export_dir / f'{SUBJECT_ID}_{SESSION_ID}_stimulus_channel_audit.csv', index=False, encoding='utf-8-sig')
onset_events.to_csv(export_dir / f'{SUBJECT_ID}_{SESSION_ID}_stimulus_onset_values.csv', index=False, encoding='utf-8-sig')
print('Audit tables exported to:', export_dir)


Channel audit:


,file,channel,role,min,max,std,nonzero_count,nonzero_fraction,unique_count,unique_values_first,transition_count,all_zero,values_at_onset_first
0,F_03_SE001_CBCol_Train01.edf,StimulusBegin,core label,0.0,1.0,0.310849,6048,0.108372,2,"[0.0, 1.0]",1512,False,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ..."
1,F_03_SE001_CBCol_Train01.edf,StimulusType,core label,0.0,1.0,0.258898,4032,0.072248,2,"[0.0, 1.0]",168,False,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,F_03_SE001_CBCol_Train01.edf,StimulusCode,sequence/phase,0.0,756.0,252.093964,36288,0.650229,757,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...",1512,False,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ..."
3,F_03_SE001_CBCol_Train01.edf,CurrentTarget,core label,0.0,21.0,8.244724,48384,0.866972,6,"[0.0, 5.0, 6.0, 18.0, 20.0, 21.0]",12,False,"[6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, ..."
4,F_03_SE001_CBCol_Train01.edf,PhaseInSequence,sequence/phase,0.0,3.0,0.493050,53760,0.963303,4,"[0.0, 1.0, 2.0, 3.0]",19,False,"[2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, ..."
5,F_03_SE001_CBCol_Train01.edf,SelectedTarget,feedback/result,0.0,31.0,7.083256,4384,0.078555,6,"[0.0, 13.0, 25.0, 27.0, 29.0, 31.0]",12,False,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
6,F_03_SE001_CBCol_Train01.edf,SelectedRow,feedback/result,0.0,6.0,1.323871,4384,0.078555,4,"[0.0, 3.0, 5.0, 6.0]",12,False,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
7,F_03_SE001_CBCol_Train01.edf,SelectedColumn,feedback/result,0.0,5.0,0.858154,4384,0.078555,4,"[0.0, 1.0, 3.0, 5.0]",12,False,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
8,F_03_SE001_CBCol_Train01.edf,DisplayResults,feedback/result,0.0,0.0,0.000000,0,0.000000,1,[0.0],0,True,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9,F_03_SE001_CBCol_Train01.edf,FakeFeedback,feedback/result,0.0,0.0,0.000000,0,0.000000,1,[0.0],0,True,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


All-zero channels by file:


,file,channel,role
8,F_03_SE001_CBCol_Train01.edf,DisplayResults,feedback/result
9,F_03_SE001_CBCol_Train01.edf,FakeFeedback,feedback/result
18,F_03_SE001_CBCol_Train02.edf,DisplayResults,feedback/result
19,F_03_SE001_CBCol_Train02.edf,FakeFeedback,feedback/result
28,F_03_SE001_CBCol_Train03.edf,DisplayResults,feedback/result
29,F_03_SE001_CBCol_Train03.edf,FakeFeedback,feedback/result


Values at first stimulus onsets:


,file,sample,time_sec,StimulusBegin,StimulusType,StimulusCode,CurrentTarget,PhaseInSequence,SelectedTarget,SelectedRow,SelectedColumn,DisplayResults,FakeFeedback
0,F_03_SE001_CBCol_Train01.edf,1152,4.50,1.0,0.0,1.0,6.0,2.0,0.0,0.0,0.0,0.0,0.0
1,F_03_SE001_CBCol_Train01.edf,1216,4.75,1.0,0.0,2.0,6.0,2.0,0.0,0.0,0.0,0.0,0.0
2,F_03_SE001_CBCol_Train01.edf,1280,5.00,1.0,0.0,3.0,6.0,2.0,0.0,0.0,0.0,0.0,0.0
3,F_03_SE001_CBCol_Train01.edf,1344,5.25,1.0,1.0,4.0,6.0,2.0,0.0,0.0,0.0,0.0,0.0
4,F_03_SE001_CBCol_Train01.edf,1408,5.50,1.0,0.0,5.0,6.0,2.0,0.0,0.0,0.0,0.0,0.0
5,F_03_SE001_CBCol_Train01.edf,1472,5.75,1.0,0.0,6.0,6.0,2.0,0.0,0.0,0.0,0.0,0.0
6,F_03_SE001_CBCol_Train01.edf,1536,6.00,1.0,0.0,7.0,6.0,2.0,0.0,0.0,0.0,0.0,0.0
7,F_03_SE001_CBCol_Train01.edf,1600,6.25,1.0,0.0,8.0,6.0,2.0,0.0,0.0,0.0,0.0,0.0
8,F_03_SE001_CBCol_Train01.edf,1664,6.50,1.0,0.0,9.0,6.0,2.0,0.0,0.0,0.0,0.0,0.0
9,F_03_SE001_CBCol_Train01.edf,1728,6.75,1.0,0.0,10.0,6.0,2.0,0.0,0.0,0.0,0.0,0.0


Audit tables exported to: d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection


## 9. 全组 Train/CBCol 的 EEG 数据统计

本节统计研究范围内所有被试和 Session 的 `Train/CBCol/*.edf` 文件。

这里只纳入 MNE 通道类型为 `eeg` 的通道。`stim`、`misc`、`eog` 等通道不会进入 EEG 数值统计，因此 `StimulusBegin`、`StimulusType`、`StimulusCode`、`CurrentTarget` 等事件通道不会污染下面的均值、标准差和幅值范围。

每个 EDF 文件单独读取和统计，避免把不同 Session 的数据直接拼接。

In [11]:
GROUP_TRAIN_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'bigP3BCI-data' / 'StudyF'
GROUP_TRAIN_FILES = sorted(GROUP_TRAIN_ROOT.glob('*/SE*/Train/CBCol/*.edf'))

if not GROUP_TRAIN_FILES:
    raise FileNotFoundError(f'No Train/CBCol EDF files found under: {GROUP_TRAIN_ROOT}')

group_file_rows = []
group_channel_rows = []

for path in GROUP_TRAIN_FILES:
    raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
    raw = assign_bigp3bci_channel_types(raw)
    eeg_names = [name for name, kind in zip(raw.ch_names, raw.get_channel_types()) if kind == 'eeg']

    if not eeg_names:
        print('WARNING: no EEG channels:', path.name)
        continue

    # MNE returns EEG data in volts; convert only the copied statistics to microvolts.
    eeg_data_uV = raw.get_data(picks=eeg_names, verbose=False) * 1e6
    subject = path.parts[-5]
    session = path.parts[-4]

    group_file_rows.append({
        'subject': subject,
        'session': session,
        'file': path.name,
        'n_eeg_channels': len(eeg_names),
        'sfreq_hz': raw.info['sfreq'],
        'duration_sec': raw.times[-1],
        'eeg_mean_uV': float(eeg_data_uV.mean()),
        'eeg_std_uV': float(eeg_data_uV.std()),
        'eeg_min_uV': float(eeg_data_uV.min()),
        'eeg_max_uV': float(eeg_data_uV.max()),
        'n_flat_eeg_channels': int(np.sum(np.ptp(eeg_data_uV, axis=1) == 0)),
    })

    for name, values in zip(eeg_names, eeg_data_uV):
        group_channel_rows.append({
            'subject': subject,
            'session': session,
            'file': path.name,
            'channel': name,
            'mne_type': 'eeg',
            'sfreq_hz': raw.info['sfreq'],
            'mean_uV': float(values.mean()),
            'std_uV': float(values.std()),
            'min_uV': float(values.min()),
            'max_uV': float(values.max()),
            'p01_uV': float(np.percentile(values, 1)),
            'p99_uV': float(np.percentile(values, 99)),
            'flat': bool(np.ptp(values) == 0),
        })

group_file_stats = pd.DataFrame(group_file_rows)
group_eeg_stats = pd.DataFrame(group_channel_rows)

print(f'Files included: {len(group_file_stats)}')
print(f'EEG channel records: {len(group_eeg_stats)}')
print(f'Unique EEG channel names: {group_eeg_stats["channel"].nunique()}')
print("All rows above have mne_type == 'eeg'.")

display(group_file_stats)
display(group_eeg_stats.head(20))

print('EEG channel counts by file:')
display(group_file_stats.groupby(['subject', 'session'])['n_eeg_channels'].agg(['count', 'min', 'max']))

print('Flat EEG channels:')
display(group_eeg_stats[group_eeg_stats['flat']])

group_export_dir = PROJECT_ROOT / 'outputs' / 'raw_inspection'
group_export_dir.mkdir(parents=True, exist_ok=True)
group_file_stats.to_csv(group_export_dir / 'StudyF_Train_CBCol_EEG_file_statistics.csv', index=False, encoding='utf-8-sig')
group_eeg_stats.to_csv(group_export_dir / 'StudyF_Train_CBCol_EEG_channel_statistics.csv', index=False, encoding='utf-8-sig')
print('Saved:', group_export_dir / 'StudyF_Train_CBCol_EEG_file_statistics.csv')
print('Saved:', group_export_dir / 'StudyF_Train_CBCol_EEG_channel_statistics.csv')

C:\Users\78431\AppData\Local\Temp\ipykernel_34568\1378353222.py:11: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
C:\Users\78431\AppData\Local\Temp\ipykernel_34568\1378353222.py:11: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
C:\Users\78431\AppData\Local\Temp\ipykernel_34568\1378353222.py:11: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
C:\Users\78431\AppData\Local\Temp\ipykernel_34568\1378353222.py:11: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
C:\U

Files included: 90
EEG channel records: 1440
Unique EEG channel names: 16
All rows above have mne_type == 'eeg'.


,subject,session,file,n_eeg_channels,sfreq_hz,duration_sec,eeg_mean_uV,eeg_std_uV,eeg_min_uV,eeg_max_uV,n_flat_eeg_channels
0,F_03,SE001,F_03_SE001_CBCol_Train01.edf,16,256.0,217.996094,-0.001956,8.039261,-61.098,73.266,0
1,F_03,SE001,F_03_SE001_CBCol_Train02.edf,16,256.0,217.996094,0.006109,7.009301,-49.698,58.332,0
2,F_03,SE001,F_03_SE001_CBCol_Train03.edf,16,256.0,217.996094,0.006367,7.576181,-64.992,65.868,0
3,F_03,SE002,F_03_SE002_CBCol_Train01.edf,16,256.0,217.996094,-0.019061,10.221325,-53.904,54.798,0
4,F_03,SE002,F_03_SE002_CBCol_Train02.edf,16,256.0,217.996094,0.010354,9.648871,-68.310,71.850,0
5,F_03,SE002,F_03_SE002_CBCol_Train03.edf,16,256.0,217.996094,0.011432,9.279344,-76.260,69.036,0
6,F_03,SE003,F_03_SE003_CBCol_Train01.edf,16,256.0,217.996094,-0.001965,7.829573,-72.048,69.306,0
7,F_03,SE003,F_03_SE003_CBCol_Train02.edf,16,256.0,217.996094,0.055016,8.367440,-84.150,99.048,0
8,F_03,SE003,F_03_SE003_CBCol_Train03.edf,16,256.0,217.996094,0.001784,8.261733,-182.910,144.102,0
9,F_05,SE001,F_05_SE001_CBCol_Train02.edf,16,256.0,217.996094,-0.079442,17.620839,-196.596,175.506,0


,subject,session,file,channel,mne_type,sfreq_hz,mean_uV,std_uV,min_uV,max_uV,p01_uV,p99_uV,flat
0,F_03,SE001,F_03_SE001_CBCol_Train01.edf,EEG_F3,eeg,256.0,-0.011130,10.303620,-47.904,62.868,-25.00716,24.70758,False
1,F_03,SE001,F_03_SE001_CBCol_Train01.edf,EEG_Fz,eeg,256.0,-0.013726,9.825145,-46.638,49.056,-23.73558,23.41716,False
2,F_03,SE001,F_03_SE001_CBCol_Train01.edf,EEG_F4,eeg,256.0,-0.016030,9.235256,-57.072,55.980,-22.39632,22.14558,False
3,F_03,SE001,F_03_SE001_CBCol_Train01.edf,EEG_T7,eeg,256.0,-0.002456,9.148488,-61.098,73.266,-21.58800,21.66558,False
4,F_03,SE001,F_03_SE001_CBCol_Train01.edf,EEG_C3,eeg,256.0,-0.002460,8.462070,-35.352,39.888,-19.91874,20.03874,False
5,F_03,SE001,F_03_SE001_CBCol_Train01.edf,EEG_Cz,eeg,256.0,-0.002112,8.181990,-35.850,37.092,-19.27200,19.39116,False
6,F_03,SE001,F_03_SE001_CBCol_Train01.edf,EEG_C4,eeg,256.0,-0.007217,6.994290,-34.998,30.996,-16.37316,16.76400,False
7,F_03,SE001,F_03_SE001_CBCol_Train01.edf,EEG_T8,eeg,256.0,-0.006022,6.728223,-30.288,29.064,-15.75558,15.96558,False
8,F_03,SE001,F_03_SE001_CBCol_Train01.edf,EEG_CP3,eeg,256.0,0.001662,9.079924,-34.344,37.248,-20.64558,21.79200,False
9,F_03,SE001,F_03_SE001_CBCol_Train01.edf,EEG_CP4,eeg,256.0,-0.001175,6.585117,-31.806,29.514,-15.30000,15.98316,False


EEG channel counts by file:


count  min  max
subject session                 
F_03    SE001        3   16   16
        SE002        3   16   16
        SE003        3   16   16
F_05    SE001        3   16   16
        SE002        3   16   16
        SE003        3   16   16
F_06    SE001        3   16   16
        SE002        3   16   16
        SE003        3   16   16
F_07    SE001        3   16   16
        SE002        3   16   16
        SE003        3   16   16
F_08    SE001        3   16   16
        SE002        3   16   16
        SE003        3   16   16
F_20    SE001        3   16   16
        SE002        3   16   16
        SE003        3   16   16
F_21    SE001        3   16   16
        SE002        3   16   16
        SE003        3   16   16
F_23    SE001        3   16   16
        SE002        3   16   16
        SE003        3   16   16
F_24    SE001        3   16   16
        SE002        3   16   16
        SE003        3   16   16
F_25    SE001        3   16   16
        SE002        3   16   16
        SE003        3   16   16

Flat EEG channels:


,subject,session,file,channel,mne_type,sfreq_hz,mean_uV,std_uV,min_uV,max_uV,p01_uV,p99_uV,flat


Saved: d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection\StudyF_Train_CBCol_EEG_file_statistics.csv
Saved: d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection\StudyF_Train_CBCol_EEG_channel_statistics.csv


## 10. 全组 EEG 频谱：不同 Subject 与总体平均

下面沿用前面的 PSD 图，但将所有 `Train/CBCol` 文件汇总。不同颜色表示不同 Subject，黑色粗线表示全组平均频谱。

计算时只选择 MNE 类型为 `eeg` 的通道，不包含任何 `stim` 或 `misc` 通道。每个文件先对其 EEG 通道求平均，再在文件层面求平均，避免记录较长的文件对总体结果产生过大影响。

In [12]:
subject_session_psd = {}
all_file_psd = []
psd_freqs = None

for path in GROUP_TRAIN_FILES:
    raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
    raw = assign_bigp3bci_channel_types(raw)
    eeg_names = [name for name, kind in zip(raw.ch_names, raw.get_channel_types()) if kind == 'eeg']
    if not eeg_names:
        continue

    spectrum = raw.compute_psd(
        fmin=0.1,
        fmax=min(125, raw.info['sfreq'] / 2 - 1),
        picks=eeg_names,
        reject_by_annotation=False,
        exclude='bads',
        verbose=False,
    )
    freqs = spectrum.freqs
    power = spectrum.get_data().mean(axis=0)
    power_db_uV = 10 * np.log10(np.maximum(power * 1e12, np.finfo(float).tiny))

    if psd_freqs is None:
        psd_freqs = freqs
    subject = path.parts[-5]
    session = path.parts[-4]
    subject_session_psd.setdefault((subject, session), []).append(power_db_uV)
    all_file_psd.append(power_db_uV)

subjects = sorted({subject for subject, session in subject_session_psd})
sessions = sorted({session for subject, session in subject_session_psd})
subject_colors = dict(zip(subjects, plt.cm.tab20(np.linspace(0, 1, max(len(subjects), 1)))))
fig, axes = plt.subplots(1, len(sessions), figsize=(7 * len(sessions), 5), sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, session in zip(axes, sessions):
    session_spectra = []
    for subject in subjects:
        spectra = subject_session_psd.get((subject, session), [])
        if not spectra:
            continue
        subject_mean = np.mean(spectra, axis=0)
        session_spectra.extend(spectra)
        ax.plot(psd_freqs, subject_mean, color=subject_colors[subject], linewidth=1.2, alpha=0.8, label=subject)

    if session_spectra:
        session_grand_mean = np.mean(session_spectra, axis=0)
        ax.plot(psd_freqs, session_grand_mean, color='black', linewidth=3, label=f'{session} average')
    ax.axvline(60, color='gray', linestyle='--', linewidth=1, alpha=0.8)
    ax.set_xlim(0.1, 125)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_title(session)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Power (dB/Hz, µV²)')
fig.suptitle('StudyF Train/CBCol: EEG PSD by Session', fontsize=14)
from matplotlib.lines import Line2D
subject_handles = [Line2D([0], [0], color=subject_colors[s], lw=2, label=s) for s in subjects]
subject_handles.append(Line2D([0], [0], color='black', lw=3, label='Session average'))
axes[-1].legend(handles=subject_handles, title='Subject / color', loc='upper right', fontsize=8)
plt.show()

print('Files included:', len(all_file_psd))
print('Subject-session combinations included:', sorted(subject_session_psd))

C:\Users\78431\AppData\Local\Temp\ipykernel_34568\2287822232.py:6: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
C:\Users\78431\AppData\Local\Temp\ipykernel_34568\2287822232.py:6: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
C:\Users\78431\AppData\Local\Temp\ipykernel_34568\2287822232.py:6: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
C:\Users\78431\AppData\Local\Temp\ipykernel_34568\2287822232.py:6: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
C:\Users

Files included: 90
Subject-session combinations included: [('F_03', 'SE001'), ('F_03', 'SE002'), ('F_03', 'SE003'), ('F_05', 'SE001'), ('F_05', 'SE002'), ('F_05', 'SE003'), ('F_06', 'SE001'), ('F_06', 'SE002'), ('F_06', 'SE003'), ('F_07', 'SE001'), ('F_07', 'SE002'), ('F_07', 'SE003'), ('F_08', 'SE001'), ('F_08', 'SE002'), ('F_08', 'SE003'), ('F_20', 'SE001'), ('F_20', 'SE002'), ('F_20', 'SE003'), ('F_21', 'SE001'), ('F_21', 'SE002'), ('F_21', 'SE003'), ('F_23', 'SE001'), ('F_23', 'SE002'), ('F_23', 'SE003'), ('F_24', 'SE001'), ('F_24', 'SE002'), ('F_24', 'SE003'), ('F_25', 'SE001'), ('F_25', 'SE002'), ('F_25', 'SE003')]


## 11. Duke 与 ETSU 两个采集机构的 EEG PSD 对比

本图按照当前的推断映射分为两个采集机构：Duke（`F_05`–`F_08`）和 ETSU（`F_03`、`F_20`、`F_21`、`F_23`、`F_24`、`F_25`）。行表示机构，列表示 Session。每条彩色曲线代表一个 Subject，黑色粗线表示该机构在该 Session 内的平均 PSD。

In [19]:
duke_subjects = {'F_05', 'F_06', 'F_07', 'F_08'}
etsu_subjects = {'F_03', 'F_20', 'F_21', 'F_23', 'F_24', 'F_25'}
available_subjects = {subject for subject, session in subject_session_psd}
group_subjects = {
    'Duke (inferred)': sorted(available_subjects & duke_subjects),
    'ETSU (inferred)': sorted(available_subjects & etsu_subjects),
}

fig, axes = plt.subplots(2, len(sessions), figsize=(7 * len(sessions), 9), sharex=True, sharey=True, constrained_layout=True)
axes = np.asarray(axes)
subject_colors = dict(zip(sorted(available_subjects), plt.cm.tab20(np.linspace(0, 1, max(len(available_subjects), 1)))))

for row, (group_name, subjects_in_group) in enumerate(group_subjects.items()):
    for col, session in enumerate(sessions):
        ax = axes[row, col]
        session_spectra = []
        for subject in subjects_in_group:
            spectra = subject_session_psd.get((subject, session), [])
            if not spectra:
                continue
            subject_mean = np.mean(spectra, axis=0)
            session_spectra.extend(spectra)
            ax.plot(psd_freqs, subject_mean, color=subject_colors[subject], linewidth=1.2, alpha=0.8, label=subject)

        if session_spectra:
            group_mean = np.mean(session_spectra, axis=0)
            ax.plot(psd_freqs, group_mean, color='black', linewidth=3, label='Group average')
        ax.axvline(60, color='gray', linestyle='--', linewidth=1, alpha=0.7)
        ax.set_xlim(0.1, 125)
        ax.grid(alpha=0.3)
        if row == 0:
            ax.set_title(session)
        if col == 0:
            ax.set_ylabel(f'{group_name}\nPower (dB/Hz, µV²)')
        if row == 1:
            ax.set_xlabel('Frequency (Hz)')
        ax.legend(fontsize=8)

fig.suptitle('StudyF Train/CBCol: EEG PSD comparison of Subject groups', fontsize=14)
plt.show()

print('Duke subjects included:', group_subjects['Duke (inferred)'])
print('ETSU subjects included:', group_subjects['ETSU (inferred)'])

Duke subjects included: ['F_05', 'F_06', 'F_07', 'F_08']
ETSU subjects included: ['F_03', 'F_20', 'F_21', 'F_23', 'F_24', 'F_25']


## 12. 审计每个 Train/Test 文件中的实际字符数量

本节遍历 StudyF 的所有 EDF，分别统计 `Train/CBCol`、`Test/Static`、`Test/Dyn` 和 `Test/DynBigram`。

字符数不使用 `CurrentTarget` 的 unique 数量，因为同一个目标字符可能在不同字符位置重复出现。这里使用刺激时刻的 `CurrentTarget` 连续目标区块估计字符边界，同时报告 flash 数和 `StimulusType` 标签数量，便于核对。

In [14]:
ALL_STUDY_F_FILES = sorted((PROJECT_ROOT / 'data' / 'raw' / 'bigP3BCI-data' / 'StudyF').rglob('*.edf'))

def rising_onsets(values, threshold=0.5):
    binary = np.asarray(values) > threshold
    return np.flatnonzero(binary & ~np.r_[False, binary[:-1]])

def count_nonzero_runs(values):
    values = np.asarray(values)
    nonzero = values != 0
    return int(np.sum(nonzero & ~np.r_[False, nonzero[:-1]]))

character_audit_rows = []
for edf_path in ALL_STUDY_F_FILES:
    raw_header = mne.io.read_raw_edf(edf_path, preload=False, infer_types=True, verbose='ERROR')
    available = set(raw_header.ch_names)
    required = {'StimulusBegin', 'CurrentTarget'}
    if not required.issubset(available):
        print('Skipped: missing required event channels:', edf_path.name)
        continue

    picks = ['StimulusBegin', 'CurrentTarget']
    if 'StimulusType' in available:
        picks.append('StimulusType')
    if 'SelectedTarget' in available:
        picks.append('SelectedTarget')
    event_data = raw_header.get_data(picks=picks, verbose=False)
    values = dict(zip(picks, event_data))
    flash_onsets = rising_onsets(values['StimulusBegin'])
    target_at_flash = values['CurrentTarget'][flash_onsets]
    valid_target = target_at_flash != 0
    target_block_starts = valid_target & ~np.r_[False, valid_target[:-1]]
    target_changes = valid_target & np.r_[True, target_at_flash[1:] != target_at_flash[:-1]] if len(target_at_flash) else np.array([], dtype=bool)

    selected_target_events = (
        count_nonzero_runs(values['SelectedTarget'])
        if 'SelectedTarget' in values else np.nan
    )
    character_count = (
        selected_target_events
        if np.isfinite(selected_target_events) and selected_target_events > 0
        else int(target_changes.sum())
    )

    row = {
        'subject': edf_path.parts[-5],
        'session': edf_path.parts[-4],
        'split': edf_path.parts[-3],
        'condition': edf_path.parts[-2],
        'file': edf_path.name,
        'n_flash_onsets': len(flash_onsets),
        'n_current_target_nonzero_blocks': int(target_block_starts.sum()),
        'n_current_target_changes_at_flash': int(target_changes.sum()),
        'n_unique_current_target_values_at_flash': int(np.unique(target_at_flash[target_at_flash != 0]).size),
        'current_target_values_at_flash': np.unique(target_at_flash[target_at_flash != 0]).tolist(),
        'estimated_characters': int(character_count),
        'sfreq_hz': raw_header.info['sfreq'],
    }
    if 'StimulusType' in values:
        row['n_target_flashes'] = int(np.sum(values['StimulusType'][flash_onsets] == 1))
        row['n_non_target_flashes'] = int(np.sum(values['StimulusType'][flash_onsets] == 0))
    else:
        row['n_target_flashes'] = np.nan
        row['n_non_target_flashes'] = np.nan
    row['n_selected_target_events'] = selected_target_events
    character_audit_rows.append(row)

character_audit = pd.DataFrame(character_audit_rows)
display(character_audit)

print('Summary by subject, session, split and condition:')
summary = (
    character_audit
    .groupby(['subject', 'session', 'split', 'condition'], as_index=False)
    .agg(
        n_files=('file', 'count'),
        estimated_characters_min=('estimated_characters', 'min'),
        estimated_characters_max=('estimated_characters', 'max'),
        total_flash_onsets=('n_flash_onsets', 'sum'),
    )
)
display(summary)

character_audit_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_all_conditions_character_audit.csv'
character_summary_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_all_conditions_character_summary.csv'
character_audit.to_csv(character_audit_export, index=False, encoding='utf-8-sig')
summary.to_csv(character_summary_export, index=False, encoding='utf-8-sig')
print('Saved:', character_audit_export)
print('Saved:', character_summary_export)

,subject,session,split,condition,file,n_flash_onsets,n_current_target_nonzero_blocks,n_current_target_changes_at_flash,n_unique_current_target_values_at_flash,current_target_values_at_flash,estimated_characters,sfreq_hz,n_target_flashes,n_non_target_flashes,n_selected_target_events
0,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,148,1,6,6,"[5.0, 7.0, 8.0, 14.0, 15.0, 21.0]",6,256.0,17,131,6
1,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test02.edf,120,1,6,6,"[1.0, 5.0, 9.0, 12.0, 19.0, 25.0]",6,256.0,13,107,6
2,F_03,SE001,Test,DynBigram,F_03_SE001_DynBigram_Test01.edf,109,1,6,6,"[5.0, 7.0, 9.0, 14.0, 15.0, 18.0]",6,256.0,13,96,6
3,F_03,SE001,Test,DynBigram,F_03_SE001_DynBigram_Test02.edf,147,1,6,5,"[4.0, 9.0, 14.0, 15.0, 23.0]",6,256.0,17,130,6
4,F_03,SE001,Test,Static,F_03_SE001_Static_Test01.edf,756,1,4,4,"[3.0, 5.0, 6.0, 15.0]",6,256.0,84,672,6
5,F_03,SE001,Test,Static,F_03_SE001_Static_Test02.edf,756,1,5,4,"[3.0, 5.0, 6.0, 20.0]",6,256.0,84,672,6
6,F_03,SE001,Train,CBCol,F_03_SE001_CBCol_Train01.edf,756,1,6,5,"[5.0, 6.0, 18.0, 20.0, 21.0]",6,256.0,84,672,6
7,F_03,SE001,Train,CBCol,F_03_SE001_CBCol_Train02.edf,756,1,6,5,"[3.0, 4.0, 5.0, 9.0, 22.0]",6,256.0,84,672,6
8,F_03,SE001,Train,CBCol,F_03_SE001_CBCol_Train03.edf,756,1,5,4,"[4.0, 5.0, 7.0, 18.0]",6,256.0,84,672,6
9,F_03,SE002,Test,Dyn,F_03_SE002_Dyn_Test01.edf,236,1,6,6,"[2.0, 4.0, 5.0, 8.0, 9.0, 14.0]",6,256.0,27,209,6


Summary by subject, session, split and condition:


,subject,session,split,condition,n_files,estimated_characters_min,estimated_characters_max,total_flash_onsets
0,F_03,SE001,Test,Dyn,2,6,6,268
1,F_03,SE001,Test,DynBigram,2,6,6,256
2,F_03,SE001,Test,Static,2,6,6,1512
3,F_03,SE001,Train,CBCol,3,6,6,2268
4,F_03,SE002,Test,Dyn,2,6,6,426
5,F_03,SE002,Test,DynBigram,2,6,6,409
6,F_03,SE002,Test,Static,2,6,6,1512
7,F_03,SE002,Train,CBCol,3,6,6,2268
8,F_03,SE003,Test,Dyn,2,6,6,354
9,F_03,SE003,Test,DynBigram,2,6,6,298


Saved: d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection\StudyF_all_conditions_character_audit.csv
Saved: d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection\StudyF_all_conditions_character_summary.csv


## 13. 审计 FakeFeedback 与 DisplayResults 通道

本节只读取两个状态通道，不加载 EEG 数据。逐个 EDF 检查它们是否全为 0，或者只在某些 Train/Test 条件中有实际记录。

In [15]:
FEEDBACK_CHANNELS = ['FakeFeedback', 'DisplayResults']
feedback_rows = []

for edf_path in ALL_STUDY_F_FILES:
    raw_header = mne.io.read_raw_edf(edf_path, preload=False, infer_types=True, verbose='ERROR')
    present = [name for name in FEEDBACK_CHANNELS if name in raw_header.ch_names]
    if not present:
        feedback_rows.append({
            'subject': edf_path.parts[-5],
            'session': edf_path.parts[-4],
            'split': edf_path.parts[-3],
            'condition': edf_path.parts[-2],
            'file': edf_path.name,
            'channel': None,
            'present': False,
        })
        continue

    data = raw_header.get_data(picks=present, verbose=False)
    for channel, values in zip(present, data):
        unique_values, counts = np.unique(values, return_counts=True)
        feedback_rows.append({
            'subject': edf_path.parts[-5],
            'session': edf_path.parts[-4],
            'split': edf_path.parts[-3],
            'condition': edf_path.parts[-2],
            'file': edf_path.name,
            'channel': channel,
            'present': True,
            'min': float(values.min()),
            'max': float(values.max()),
            'std': float(values.std()),
            'nonzero_count': int(np.count_nonzero(values)),
            'unique_count': int(len(unique_values)),
            'unique_values': unique_values.tolist(),
            'all_zero': bool(np.all(values == 0)),
        })

feedback_audit = pd.DataFrame(feedback_rows)
display(feedback_audit)

print('Summary by split and condition:')
feedback_summary = (
    feedback_audit[feedback_audit['present'] == True]
    .groupby(['split', 'condition', 'channel'], as_index=False)
    .agg(
        n_files=('file', 'count'),
        n_all_zero=('all_zero', 'sum'),
        max_value=('max', 'max'),
        total_nonzero_samples=('nonzero_count', 'sum'),
    )
)
display(feedback_summary)

feedback_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_feedback_display_audit.csv'
feedback_summary_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_feedback_display_summary.csv'
feedback_audit.to_csv(feedback_export, index=False, encoding='utf-8-sig')
feedback_summary.to_csv(feedback_summary_export, index=False, encoding='utf-8-sig')
print('Saved:', feedback_export)
print('Saved:', feedback_summary_export)

,subject,session,split,condition,file,channel,present,min,max,std,nonzero_count,unique_count,unique_values,all_zero
0,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,FakeFeedback,True,0.0,0.0,0.000000,0,1,[0.0],True
1,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,DisplayResults,True,0.0,1.0,0.445362,4608,2,"[0.0, 1.0]",False
2,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test02.edf,FakeFeedback,True,0.0,0.0,0.000000,0,1,[0.0],True
3,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test02.edf,DisplayResults,True,0.0,1.0,0.460443,4608,2,"[0.0, 1.0]",False
4,F_03,SE001,Test,DynBigram,F_03_SE001_DynBigram_Test01.edf,FakeFeedback,True,0.0,0.0,0.000000,0,1,[0.0],True
5,F_03,SE001,Test,DynBigram,F_03_SE001_DynBigram_Test01.edf,DisplayResults,True,0.0,1.0,0.466476,4608,2,"[0.0, 1.0]",False
6,F_03,SE001,Test,DynBigram,F_03_SE001_DynBigram_Test02.edf,FakeFeedback,True,0.0,0.0,0.000000,0,1,[0.0],True
7,F_03,SE001,Test,DynBigram,F_03_SE001_DynBigram_Test02.edf,DisplayResults,True,0.0,1.0,0.445889,4608,2,"[0.0, 1.0]",False
8,F_03,SE001,Test,Static,F_03_SE001_Static_Test01.edf,FakeFeedback,True,0.0,0.0,0.000000,0,1,[0.0],True
9,F_03,SE001,Test,Static,F_03_SE001_Static_Test01.edf,DisplayResults,True,0.0,1.0,0.275229,4608,2,"[0.0, 1.0]",False


Summary by split and condition:


,split,condition,channel,n_files,n_all_zero,max_value,total_nonzero_samples
0,Test,Dyn,DisplayResults,60,0,1.0,275712
1,Test,Dyn,FakeFeedback,60,60,0.0,0
2,Test,DynBigram,DisplayResults,60,2,1.0,267264
3,Test,DynBigram,FakeFeedback,60,60,0.0,0
4,Test,Static,DisplayResults,60,0,1.0,276480
5,Test,Static,FakeFeedback,60,60,0.0,0
6,Train,CBCol,DisplayResults,90,90,0.0,0
7,Train,CBCol,FakeFeedback,90,90,0.0,0


Saved: d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection\StudyF_feedback_display_audit.csv
Saved: d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection\StudyF_feedback_display_summary.csv


## 14. 单个 6-character Train 文件的 EEG 与事件核对

本节暂不滤波、不切 epoch、不改变原始数据。先核对 EEG 单位与参考信息，并检查 `StimulusBegin`、`StimulusType`、`CurrentTarget`、字符状态通道之间的关系。

In [16]:
AUDIT_FILE = PROJECT_ROOT / 'data' / 'raw' / 'bigP3BCI-data' / 'StudyF' / 'F_03' / 'SE001' / 'Train' / 'CBCol' / 'F_03_SE001_CBCol_Train01.edf'
audit_raw = mne.io.read_raw_edf(AUDIT_FILE, preload=False, infer_types=True, verbose='ERROR')
audit_raw = assign_bigp3bci_channel_types(audit_raw)

EEG_NAMES = [name for name in audit_raw.ch_names if name.upper().startswith('EEG_')]
CHARACTER_NAMES = [name for name in audit_raw.ch_names if re.match(r'^.+_\d+_\d+$', name)]
EVENT_NAMES = ['StimulusBegin', 'StimulusType', 'StimulusCode', 'CurrentTarget', 'SelectedTarget']
EVENT_NAMES = [name for name in EVENT_NAMES if name in audit_raw.ch_names]
PICKS_FOR_AUDIT = EEG_NAMES + CHARACTER_NAMES + EVENT_NAMES
audit_data = audit_raw.get_data(picks=PICKS_FOR_AUDIT, verbose=False)
audit_values = dict(zip(PICKS_FOR_AUDIT, audit_data))

print('File:', AUDIT_FILE.name)
print('Sampling frequency:', audit_raw.info['sfreq'], 'Hz')
print('Duration:', audit_raw.times[-1], 's')
print('EEG channels:', len(EEG_NAMES))

unit_rows = []
for name in EEG_NAMES:
    idx = audit_raw.ch_names.index(name)
    ch_info = audit_raw.info['chs'][idx]
    unit_rows.append({
        'channel': name,
        'mne_unit_code': ch_info['unit'],
        'unit_multiplier': ch_info['unit_mul'],
        'raw_data_unit': 'V (MNE convention)',
        'display_unit': 'µV in MNE plots',
    })
print('EEG unit metadata:')
display(pd.DataFrame(unit_rows))
print('custom_ref_applied:', audit_raw.info.get('custom_ref_applied'))
print('montage:', audit_raw.get_montage())
print('description:', audit_raw.info.get('description'))
print('subject_info:', audit_raw.info.get('subject_info'))

def rising_edges(values, threshold=0.5):
    binary = np.asarray(values) > threshold
    return np.flatnonzero(binary & ~np.r_[False, binary[:-1]])

stimulus_onsets = rising_edges(audit_values['StimulusBegin'])
stimulus_begin_changes = np.flatnonzero(np.diff(audit_values['StimulusBegin']) != 0) + 1
print('StimulusBegin rising edges:', len(stimulus_onsets))
print('StimulusBegin all value changes:', len(stimulus_begin_changes))
print('First stimulus onset samples:', stimulus_onsets[:10].tolist())

onset_table = pd.DataFrame({
    'sample': stimulus_onsets,
    'time_sec': stimulus_onsets / audit_raw.info['sfreq'],
    'stimulus_code': audit_values['StimulusCode'][stimulus_onsets] if 'StimulusCode' in audit_values else np.nan,
    'stimulus_type': audit_values['StimulusType'][stimulus_onsets] if 'StimulusType' in audit_values else np.nan,
    'current_target': audit_values['CurrentTarget'][stimulus_onsets] if 'CurrentTarget' in audit_values else np.nan,
})
print('StimulusType values at flash onsets:', sorted(onset_table['stimulus_type'].dropna().unique().tolist()))
display(onset_table.head(20))

selected_target_onsets = rising_edges(audit_values['SelectedTarget']) if 'SelectedTarget' in audit_values else np.array([], dtype=int)
print('SelectedTarget events:', len(selected_target_onsets))

if len(selected_target_onsets):
    character_rows = []
    previous_boundary = -1
    for character_index, boundary in enumerate(selected_target_onsets, start=1):
        flash_mask = (stimulus_onsets > previous_boundary) & (stimulus_onsets < boundary)
        flash_indices = np.flatnonzero(flash_mask)
        target_values = onset_table.loc[flash_indices, 'stimulus_type'].to_numpy()
        current_values = onset_table.loc[flash_indices, 'current_target'].to_numpy()
        character_rows.append({
            'character_trial': character_index,
            'n_flashes': len(flash_indices),
            'n_target_epochs': int(np.sum(target_values == 1)),
            'n_non_target_epochs': int(np.sum(target_values == 0)),
            'target_label_values': sorted(np.unique(current_values).tolist()),
            'missing_stimulus_type': bool(np.any(~np.isin(target_values, [0, 1]))),
        })
        previous_boundary = boundary
    character_table = pd.DataFrame(character_rows)
    print('Character-trial audit:')
    display(character_table)
else:
    character_table = pd.DataFrame()
    print('No SelectedTarget events found; character boundaries require a separate rule.')

active_character_rows = []
for sample in stimulus_onsets:
    active = [name for name in CHARACTER_NAMES if audit_values[name][sample] > 0]
    active_character_rows.append({
        'sample': sample,
        'n_active_character_channels': len(active),
        'active_character_channels': active,
    })
active_character_table = pd.DataFrame(active_character_rows)
print('Character-channel activity at first flash onsets:')
display(active_character_table.head(20))

print('Missing stimulus labels:', int(onset_table['stimulus_type'].isna().sum()))
print('Invalid StimulusType values:', int((~onset_table['stimulus_type'].isin([0, 1])).sum()))
print('Flash-onset sample duplicates:', int(onset_table['sample'].duplicated().sum()))
print('StimulusCode duplicate values at adjacent flashes:', int(np.sum(np.diff(onset_table['stimulus_code']) == 0)))


File: F_03_SE001_CBCol_Train01.edf
Sampling frequency: 256.0 Hz
Duration: 217.99609375 s
EEG channels: 16
EEG unit metadata:


,channel,mne_unit_code,unit_multiplier,raw_data_unit,display_unit
0,EEG_F3,107,0,V (MNE convention),µV in MNE plots
1,EEG_Fz,107,0,V (MNE convention),µV in MNE plots
2,EEG_F4,107,0,V (MNE convention),µV in MNE plots
3,EEG_T7,107,0,V (MNE convention),µV in MNE plots
4,EEG_C3,107,0,V (MNE convention),µV in MNE plots
5,EEG_Cz,107,0,V (MNE convention),µV in MNE plots
6,EEG_C4,107,0,V (MNE convention),µV in MNE plots
7,EEG_T8,107,0,V (MNE convention),µV in MNE plots
8,EEG_CP3,107,0,V (MNE convention),µV in MNE plots
9,EEG_CP4,107,0,V (MNE convention),µV in MNE plots


custom_ref_applied: 0 (FIFFV_MNE_CUSTOM_REF_OFF)
montage: None
description: None
subject_info: <subject_info | his_id: F_03, sex: 2, first_name: X, last_name: X, birthday: 1976-01-01>
StimulusBegin rising edges: 756
StimulusBegin all value changes: 1512
First stimulus onset samples: [1152, 1216, 1280, 1344, 1408, 1472, 1536, 1600, 1664, 1728]
StimulusType values at flash onsets: [0.0, 1.0]


,sample,time_sec,stimulus_code,stimulus_type,current_target
0,1152,4.50,1.0,0.0,6.0
1,1216,4.75,2.0,0.0,6.0
2,1280,5.00,3.0,0.0,6.0
3,1344,5.25,4.0,1.0,6.0
4,1408,5.50,5.0,0.0,6.0
5,1472,5.75,6.0,0.0,6.0
6,1536,6.00,7.0,0.0,6.0
7,1600,6.25,8.0,0.0,6.0
8,1664,6.50,9.0,0.0,6.0
9,1728,6.75,10.0,0.0,6.0


SelectedTarget events: 6
Character-trial audit:


,character_trial,n_flashes,n_target_epochs,n_non_target_epochs,target_label_values,missing_stimulus_type
0,1,126,14,112,[6.0],False
1,2,126,14,112,[21.0],False
2,3,126,14,112,[20.0],False
3,4,126,14,112,[21.0],False
4,5,126,14,112,[18.0],False
5,6,126,14,112,[5.0],False


Character-channel activity at first flash onsets:


,sample,n_active_character_channels,active_character_channels
0,1152,4,"[D_1_4, G_2_1, N_3_2, R_3_6]"
1,1216,4,"[S_4_1, W_4_5, Z_5_2, 4_6_1]"
2,1280,4,"[B_1_2, P_3_4, 1_5_4, 8_6_5]"
3,1344,3,"[F_1_6, U_4_3, 3_5_6]"
4,1408,3,"[I_2_3, K_2_5, 6_6_3]"
5,1472,3,"[L_2_6, Q_3_5, 7_6_4]"
6,1536,3,"[E_1_5, H_2_2, J_2_4]"
7,1600,4,"[M_3_1, V_4_4, 5_6_2, 9_6_6]"
8,1664,4,"[A_1_1, T_4_2, Sp_5_3, 2_5_5]"
9,1728,4,"[C_1_3, O_3_3, X_4_6, Y_5_1]"


Missing stimulus labels: 0
Invalid StimulusType values: 0
Flash-onset sample duplicates: 0
StimulusCode duplicate values at adjacent flashes: 0


## 15. 审计 StimulusCode 的取值范围

分别统计 `StimulusCode` 在整个通道中的非零取值范围，以及它在 `StimulusBegin` flash 上升沿时的取值范围。

In [17]:
stimulus_code_rows = []
for edf_path in ALL_STUDY_F_FILES:
    raw_header = mne.io.read_raw_edf(edf_path, preload=False, infer_types=True, verbose='ERROR')
    if not {'StimulusCode', 'StimulusBegin'}.issubset(raw_header.ch_names):
        continue
    values = raw_header.get_data(picks=['StimulusCode', 'StimulusBegin'], verbose=False)
    stimulus_code = values[0]
    stimulus_begin = values[1]
    nonzero_code = stimulus_code[stimulus_code != 0]
    onset_samples = rising_onsets(stimulus_begin)
    onset_code = stimulus_code[onset_samples]
    onset_nonzero_code = onset_code[onset_code != 0]
    stimulus_code_rows.append({
        'subject': edf_path.parts[-5],
        'session': edf_path.parts[-4],
        'split': edf_path.parts[-3],
        'condition': edf_path.parts[-2],
        'file': edf_path.name,
        'overall_min_nonzero': float(nonzero_code.min()) if len(nonzero_code) else np.nan,
        'overall_max_nonzero': float(nonzero_code.max()) if len(nonzero_code) else np.nan,
        'overall_unique_nonzero': int(np.unique(nonzero_code).size),
        'onset_min_nonzero': float(onset_nonzero_code.min()) if len(onset_nonzero_code) else np.nan,
        'onset_max_nonzero': float(onset_nonzero_code.max()) if len(onset_nonzero_code) else np.nan,
        'onset_unique_nonzero': int(np.unique(onset_nonzero_code).size),
        'n_onsets': int(len(onset_samples)),
        'n_zero_code_at_onset': int(np.sum(onset_code == 0)),
    })

stimulus_code_audit = pd.DataFrame(stimulus_code_rows)
display(stimulus_code_audit)
print('Overall range by condition:')
display(stimulus_code_audit.groupby(['split', 'condition'])[['overall_min_nonzero', 'overall_max_nonzero', 'overall_unique_nonzero']].agg(['min', 'max']))
print('Range at StimulusBegin onsets by condition:')
display(stimulus_code_audit.groupby(['split', 'condition'])[['onset_min_nonzero', 'onset_max_nonzero', 'onset_unique_nonzero', 'n_zero_code_at_onset']].agg(['min', 'max']))

stimulus_code_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_StimulusCode_range_audit.csv'
stimulus_code_audit.to_csv(stimulus_code_export, index=False, encoding='utf-8-sig')
print('Saved:', stimulus_code_export)

,subject,session,split,condition,file,overall_min_nonzero,overall_max_nonzero,overall_unique_nonzero,onset_min_nonzero,onset_max_nonzero,onset_unique_nonzero,n_onsets,n_zero_code_at_onset
0,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,1.0,180.0,136,1.0,180.0,136,148,0
1,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test02.edf,1.0,163.0,120,1.0,163.0,120,120,0
2,F_03,SE001,Test,DynBigram,F_03_SE001_DynBigram_Test01.edf,1.0,143.0,109,1.0,143.0,109,109,0
3,F_03,SE001,Test,DynBigram,F_03_SE001_DynBigram_Test02.edf,1.0,180.0,143,1.0,180.0,143,147,0
4,F_03,SE001,Test,Static,F_03_SE001_Static_Test01.edf,1.0,756.0,756,1.0,756.0,756,756,0
5,F_03,SE001,Test,Static,F_03_SE001_Static_Test02.edf,1.0,756.0,756,1.0,756.0,756,756,0
6,F_03,SE001,Train,CBCol,F_03_SE001_CBCol_Train01.edf,1.0,756.0,756,1.0,756.0,756,756,0
7,F_03,SE001,Train,CBCol,F_03_SE001_CBCol_Train02.edf,1.0,756.0,756,1.0,756.0,756,756,0
8,F_03,SE001,Train,CBCol,F_03_SE001_CBCol_Train03.edf,1.0,756.0,756,1.0,756.0,756,756,0
9,F_03,SE002,Test,Dyn,F_03_SE002_Dyn_Test01.edf,1.0,177.0,172,1.0,177.0,172,236,0


Overall range by condition:


overall_min_nonzero      overall_max_nonzero        overall_unique_nonzero     
                                min  max                 min    max                    min  max
split condition                                                                                
Test  Dyn                       1.0  1.0               124.0  847.0                    102  762
      DynBigram                 1.0  1.0               142.0  847.0                    102  762
      Static                    1.0  1.0               666.0  756.0                    666  756
Train CBCol                     1.0  1.0               738.0  756.0                    738  756

Range at StimulusBegin onsets by condition:


onset_min_nonzero      onset_max_nonzero        onset_unique_nonzero      n_zero_code_at_onset    
                              min  max               min    max                  min  max                  min max
split condition                                                                                                   
Test  Dyn                     1.0  1.0             124.0  847.0                  102  762                    0   0
      DynBigram               1.0  1.0             142.0  847.0                  102  762                    0   0
      Static                  1.0  1.0             666.0  756.0                  666  756                    0   0
Train CBCol                   1.0  1.0             738.0  756.0                  738  756                    0   0

Saved: d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection\StudyF_StimulusCode_range_audit.csv


## 16. 固定的逐 character event-QC

完整 character 的标准为 126 flashes、14 target、112 non-target、CurrentTarget 恒定、StimulusType 只有 0/1、StimulusCode 连续或合法重置、相邻 flash 间隔约为 0.25 秒。Train/CBCol 和 Test/Static 使用固定标准；Dyn 条件单独标记。

In [18]:
EXPECTED_FLASHES, EXPECTED_TARGET, EXPECTED_NON_TARGET = 126, 14, 112
EXPECTED_INTERVAL_SEC = 0.25
character_qc_rows, file_qc_rows = [], []

def classify_code_sequence(codes):
    codes = np.asarray(codes)
    if len(codes) == 0 or np.any(codes == 0):
        return 'invalid_or_zero'
    if len(codes) == 1 or np.all(np.diff(codes) == 1):
        return 'monotonic_within_character'
    return 'non_monotonic_or_unexpected'

for edf_path in ALL_STUDY_F_FILES:
    raw_header = mne.io.read_raw_edf(edf_path, preload=False, infer_types=True, verbose='ERROR')
    required = ['StimulusBegin', 'StimulusType', 'StimulusCode', 'CurrentTarget', 'SelectedTarget']
    missing = [name for name in required if name not in raw_header.ch_names]
    base = {'subject': edf_path.parts[-5], 'session': edf_path.parts[-4], 'split': edf_path.parts[-3], 'condition': edf_path.parts[-2], 'file': edf_path.name}
    if missing:
        file_qc_rows.append({**base, 'n_character_trials': 0, 'file_status': 'needs_review', 'review_notes': f'missing channels: {missing}'})
        continue
    data = raw_header.get_data(picks=required, verbose=False)
    values = dict(zip(required, data))
    flash_onsets = rising_onsets(values['StimulusBegin'])
    character_boundaries = rising_onsets(values['SelectedTarget'])
    rows_for_file = []
    previous_boundary = -1
    fixed_standard = (edf_path.parts[-3] == 'Train' and edf_path.parts[-2] == 'CBCol') or (edf_path.parts[-3] == 'Test' and edf_path.parts[-2] == 'Static')
    for character_index, boundary in enumerate(character_boundaries, start=1):
        mask = (flash_onsets > previous_boundary) & (flash_onsets < boundary)
        samples = flash_onsets[mask]
        stimulus_type = values['StimulusType'][samples]
        current_target = values['CurrentTarget'][samples]
        stimulus_code = values['StimulusCode'][samples]
        intervals = np.diff(samples) / raw_header.info['sfreq']
        target_values = current_target[current_target != 0]
        type_ok = bool(np.all(np.isin(stimulus_type, [0, 1])))
        target_constant = bool(len(target_values) > 0 and len(np.unique(target_values)) == 1)
        interval_ok = bool(len(intervals) == 0 or np.all(np.isclose(intervals, EXPECTED_INTERVAL_SEC, atol=1 / raw_header.info['sfreq'])))
        code_pattern = classify_code_sequence(stimulus_code)
        standard_ok = len(samples) == EXPECTED_FLASHES and int(np.sum(stimulus_type == 1)) == EXPECTED_TARGET and int(np.sum(stimulus_type == 0)) == EXPECTED_NON_TARGET
        structural_ok = type_ok and target_constant and interval_ok and code_pattern == 'monotonic_within_character'
        if not structural_ok:
            status = 'needs_review'
        elif fixed_standard and standard_ok:
            status = 'complete'
        elif fixed_standard:
            status = 'partial'
        else:
            status = 'needs_review'
        row = {**base, 'character_trial': character_index, 'n_flashes': len(samples), 'n_target': int(np.sum(stimulus_type == 1)), 'n_non_target': int(np.sum(stimulus_type == 0)), 'current_target_values': np.unique(target_values).tolist(), 'current_target_constant': target_constant, 'stimulus_type_values': np.unique(stimulus_type).tolist(), 'stimulus_type_only_0_1': type_ok, 'stimulus_code_start': float(stimulus_code[0]) if len(stimulus_code) else np.nan, 'stimulus_code_end': float(stimulus_code[-1]) if len(stimulus_code) else np.nan, 'stimulus_code_pattern': code_pattern, 'median_flash_interval_sec': float(np.median(intervals)) if len(intervals) else np.nan, 'interval_0_25_sec_ok': interval_ok, 'fixed_standard_applicable': fixed_standard, 'standard_counts_ok': standard_ok, 'character_status': status}
        character_qc_rows.append(row)
        rows_for_file.append(row)
        previous_boundary = boundary
    statuses = [row['character_status'] for row in rows_for_file]
    if not statuses:
        file_status, notes = 'needs_review', 'no character boundaries detected'
    elif any(status == 'needs_review' for status in statuses):
        file_status, notes = 'needs_review', 'structural issue or dynamic condition requires review'
    elif all(status == 'complete' for status in statuses):
        file_status, notes = 'complete', ''
    else:
        file_status, notes = 'partial', 'one or more characters fail the fixed count standard'
    file_qc_rows.append({**base, 'n_character_trials': len(rows_for_file), 'file_status': file_status, 'review_notes': notes})

character_event_qc = pd.DataFrame(character_qc_rows)
file_event_qc = pd.DataFrame(file_qc_rows)
display(character_event_qc)
display(file_event_qc)
display(file_event_qc.groupby(['split', 'condition', 'file_status']).size().reset_index(name='n_files'))
character_qc_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_character_event_qc.csv'
file_qc_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_file_event_qc.csv'
character_event_qc.to_csv(character_qc_export, index=False, encoding='utf-8-sig')
file_event_qc.to_csv(file_qc_export, index=False, encoding='utf-8-sig')
print('Saved:', character_qc_export)
print('Saved:', file_qc_export)

,subject,session,split,condition,file,character_trial,n_flashes,n_target,n_non_target,current_target_values,current_target_constant,stimulus_type_values,stimulus_type_only_0_1,stimulus_code_start,stimulus_code_end,stimulus_code_pattern,median_flash_interval_sec,interval_0_25_sec_ok,fixed_standard_applicable,standard_counts_ok,character_status
0,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,1,20,2,18,[5.0],True,"[0.0, 1.0]",True,1.0,20.0,monotonic_within_character,0.25,True,False,False,needs_review
1,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,2,32,4,28,[14.0],True,"[0.0, 1.0]",True,37.0,68.0,monotonic_within_character,0.25,True,False,False,needs_review
2,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,3,18,2,16,[15.0],True,"[0.0, 1.0]",True,73.0,90.0,monotonic_within_character,0.25,True,False,False,needs_review
3,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,4,23,3,20,[21.0],True,"[0.0, 1.0]",True,91.0,113.0,monotonic_within_character,0.25,True,False,False,needs_review
4,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,5,25,3,22,[7.0],True,"[0.0, 1.0]",True,127.0,151.0,monotonic_within_character,0.25,True,False,False,needs_review
5,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,6,30,3,27,[8.0],True,"[0.0, 1.0]",True,163.0,12.0,non_monotonic_or_unexpected,0.25,True,False,False,needs_review
6,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test02.edf,1,21,2,19,[5.0],True,"[0.0, 1.0]",True,1.0,21.0,monotonic_within_character,0.25,True,False,False,needs_review
7,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test02.edf,2,27,3,24,[1.0],True,"[0.0, 1.0]",True,37.0,63.0,monotonic_within_character,0.25,True,False,False,needs_review
8,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test02.edf,3,17,2,15,[19.0],True,"[0.0, 1.0]",True,73.0,89.0,monotonic_within_character,0.25,True,False,False,needs_review
9,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test02.edf,4,15,2,13,[9.0],True,"[0.0, 1.0]",True,91.0,105.0,monotonic_within_character,0.25,True,False,False,needs_review


,subject,session,split,condition,file,n_character_trials,file_status,review_notes
0,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test01.edf,6,needs_review,structural issue or dynamic condition requires...
1,F_03,SE001,Test,Dyn,F_03_SE001_Dyn_Test02.edf,6,needs_review,structural issue or dynamic condition requires...
2,F_03,SE001,Test,DynBigram,F_03_SE001_DynBigram_Test01.edf,6,needs_review,structural issue or dynamic condition requires...
3,F_03,SE001,Test,DynBigram,F_03_SE001_DynBigram_Test02.edf,6,needs_review,structural issue or dynamic condition requires...
4,F_03,SE001,Test,Static,F_03_SE001_Static_Test01.edf,6,complete,
5,F_03,SE001,Test,Static,F_03_SE001_Static_Test02.edf,6,complete,
6,F_03,SE001,Train,CBCol,F_03_SE001_CBCol_Train01.edf,6,complete,
7,F_03,SE001,Train,CBCol,F_03_SE001_CBCol_Train02.edf,6,complete,
8,F_03,SE001,Train,CBCol,F_03_SE001_CBCol_Train03.edf,6,complete,
9,F_03,SE002,Test,Dyn,F_03_SE002_Dyn_Test01.edf,6,needs_review,structural issue or dynamic condition requires...


,split,condition,file_status,n_files
0,Test,Dyn,needs_review,60
1,Test,DynBigram,needs_review,60
2,Test,Static,complete,58
3,Test,Static,partial,2
4,Train,CBCol,complete,88
5,Train,CBCol,partial,2


Saved: d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection\StudyF_character_event_qc.csv
Saved: d:\BCI_project\bigp3bci-personalized-strategies\outputs\raw_inspection\StudyF_file_event_qc.csv
